# 4 - The Matrix — Solutions

This file contains worked solutions for all exercises in notebook 4.

## Setup

In [ ]:
import bw_processing as bwp
from bw_processing import MatrixEntry, MatrixName, create_datapackage_from_entries
from stats_arrays import NormalUncertainty, TriangularUncertainty
import matrix_utils as mu
import bw2calc as bc
import numpy as np
import seaborn as sb
import pandas as pd

natural_gas = 101
carbon_fibre = 102
bike = 103
co2 = 201

## Exercise 1 — Calculate s by hand

**Solution:**

Work backwards from the demand vector f = [0, 0, 1] (1 bike):

- **s[2] = 1** — bike production runs at scale 1 to satisfy the demand of 1 bike.
- **s[1] = 2.5** — carbon fibre production must supply 2.5 kg CF consumed by 1 unit of bike production (2.5 × 1 = 2.5).
- **s[0] = 237 × 2.5 = 592.5** — natural gas production must supply 237 MJ/kg × 2.5 kg CF = 592.5 MJ.

So **s = [592.5, 2.5, 1]**.

In [ ]:
A = np.array([
    [1, -237, 0],
    [0, 1, -2.5],
    [0, 0, 1],
], dtype=float)

f = np.array([0, 0, 1])  # demand: 1 bike

s = np.linalg.solve(A, f)
print("Scaling vector s:", s)
# Expected: [592.5, 2.5, 1.0]

B = np.array([[0, 26.6, 0]])  # CO2 row, process columns
g = B @ s
print("Inventory g (total CO2):", g)
# Expected: [66.5] — that's 26.6 kg CO2 per kg CF × 2.5 kg CF

C = np.array([[1]])  # CO2 characterization factor
score = C @ g
print("Score:", score)
# Expected: [[66.5]]

## Build `dp_static` and run the first LCA

In [ ]:
dp_static = create_datapackage_from_entries({
    MatrixName.technosphere: [
        MatrixEntry(row=natural_gas, col=natural_gas, amount=1),
        MatrixEntry(row=carbon_fibre, col=carbon_fibre, amount=1),
        MatrixEntry(row=bike, col=bike, amount=1),
        MatrixEntry(row=natural_gas, col=carbon_fibre, amount=237, flip=True),
        MatrixEntry(row=carbon_fibre, col=bike, amount=2.5, flip=True),
    ],
    MatrixName.biosphere: [
        MatrixEntry(row=co2, col=carbon_fibre, amount=26.6),
    ],
    MatrixName.characterization: [
        MatrixEntry(row=co2, col=co2, amount=1),
    ],
})

lca = bc.LCA(
    demand={bike: 1},
    data_objs=[dp_static],
)
lca.lci()
lca.lcia()
print("LCA score:", lca.score)

## Exercise 2 — Verify s with `lca.dicts`

The value at (natural_gas row, carbon_fibre column) in the technosphere matrix should be −237.0.

In [ ]:
# Find the row index for natural_gas in the technosphere matrix
ng_row = lca.dicts.product[natural_gas]
print(f"natural_gas row index: {ng_row}")

# Find the column index for carbon_fibre production in the technosphere matrix
cf_col = lca.dicts.activity[carbon_fibre]
print(f"carbon_fibre production column index: {cf_col}")

# Print the value at that position — should be -237.0
value = lca.technosphere_matrix[ng_row, cf_col]
print(f"technosphere_matrix[natural_gas, CF production] = {value}")
# Expected: -237.0

# Let's also look at the full technosphere matrix as a dense array
print("\nFull technosphere matrix (dense):")
print(lca.technosphere_matrix.toarray())

## Exercise 3 — Check the biosphere matrix

In [ ]:
# What is the value at (co2, carbon_fibre_production) in the biosphere matrix?
co2_row = lca.dicts.biosphere[co2]
cf_col = lca.dicts.activity[carbon_fibre]

bio_value = lca.biosphere_matrix[co2_row, cf_col]
print(f"biosphere_matrix[co2, CF production] = {bio_value}")
# Expected: 26.6

print("\nFull biosphere matrix (dense):")
print(lca.biosphere_matrix.toarray())

## Exercise 4 — Inspect the supply array

In [ ]:
print("Supply array (s):", lca.supply_array)

# Map each entry back to what process it represents
# lca.dicts.activity maps node_id -> column index
# We need the reverse: column index -> node_id
# We can build this by inverting the dict
col_to_node = {v: k for k, v in lca.dicts.activity.items()}

print("\nScaling vector with labels:")
for col_idx, scale in enumerate(lca.supply_array):
    node_id = col_to_node[col_idx]
    label = {natural_gas: 'natural_gas_production', carbon_fibre: 'CF_production', bike: 'bike_production'}.get(node_id, str(node_id))
    print(f"  {label}: {scale:.1f}")

# Verify against hand calculation:
# natural_gas_production should be 592.5
# CF_production should be 2.5
# bike_production should be 1.0
ng_col = lca.dicts.activity[natural_gas]
cf_col_idx = lca.dicts.activity[carbon_fibre]
bike_col = lca.dicts.activity[bike]
print(f"\nnatural_gas_production scale: {lca.supply_array[ng_col]:.1f}  (expected 592.5)")
print(f"CF_production scale:          {lca.supply_array[cf_col_idx]:.1f}  (expected 2.5)")
print(f"bike_production scale:         {lca.supply_array[bike_col]:.1f}  (expected 1.0)")

## Exercise 5 — Explore `lca.characterized_inventory`

In [ ]:
# Shape of inventory (B·s): rows = elementary flows, columns = processes
print("lca.inventory shape:", lca.inventory.shape)
print("lca.inventory (dense):")
print(lca.inventory.toarray())

# Shape of characterized_inventory (C·B·s = C·inventory):
# rows = impact categories, columns = processes
print("\nlca.characterized_inventory shape:", lca.characterized_inventory.shape)
print("lca.characterized_inventory (dense):")
print(lca.characterized_inventory.toarray())

# lca.score is the sum of all entries in characterized_inventory
print("\nlca.score:", lca.score)
print("sum of characterized_inventory:", lca.characterized_inventory.sum())
# They should be equal — lca.score == characterized_inventory.sum()

## Exercise 6 — Percentiles from Monte Carlo

In [ ]:
dp_stochastic = create_datapackage_from_entries({
    MatrixName.technosphere: [
        MatrixEntry(row=natural_gas, col=natural_gas, amount=1),
        MatrixEntry(row=carbon_fibre, col=carbon_fibre, amount=1),
        MatrixEntry(row=bike, col=bike, amount=1),
        MatrixEntry(row=natural_gas, col=carbon_fibre, amount=237, flip=True,
                    uncertainty_type=TriangularUncertainty.id, loc=237, minimum=200, maximum=300),
        MatrixEntry(row=carbon_fibre, col=bike, amount=2.5, flip=True,
                    uncertainty_type=TriangularUncertainty.id, loc=2.5, minimum=2, maximum=3),
    ],
    MatrixName.biosphere: [
        MatrixEntry(row=co2, col=carbon_fibre, amount=26.6,
                    uncertainty_type=NormalUncertainty.id, loc=26.6, scale=1.5),
    ],
    MatrixName.characterization: [
        MatrixEntry(row=co2, col=co2, amount=1),
    ],
})

lca = bc.LCA(
    demand={bike: 1},
    data_objs=[dp_stochastic],
    use_distributions=True,
)
lca.lci()
lca.lcia()

mc_results = [lca.score for _ in zip(range(200), lca)]
mc_array = np.array(mc_results)

p5 = np.percentile(mc_array, 5)
p95 = np.percentile(mc_array, 95)
print(f"5th percentile:  {p5:.1f} kg CO2-eq")
print(f"95th percentile: {p95:.1f} kg CO2-eq")
print(f"90% CI width: {p95 - p5:.1f} kg CO2-eq")
print(f"Mean: {mc_array.mean():.1f} kg CO2-eq")

## Exercise 7 — Sensitivity analysis with correlation coefficients

In [ ]:
# Run MC again from scratch to capture parameter values
lca = bc.LCA(
    demand={bike: 1},
    data_objs=[dp_stochastic],
    use_distributions=True,
)
lca.lci()
lca.lcia()

scores = []
ng_values = []
cf_values = []
co2_values = []

for _ in range(200):
    next(lca)
    scores.append(lca.score)
    # Capture sampled exchange values (note: technosphere stores negative for inputs)
    ng_values.append(
        -1 * lca.technosphere_matrix[lca.dicts.product[natural_gas], lca.dicts.activity[carbon_fibre]]
    )
    cf_values.append(
        -1 * lca.technosphere_matrix[lca.dicts.product[carbon_fibre], lca.dicts.activity[bike]]
    )
    co2_values.append(
        lca.biosphere_matrix[lca.dicts.biosphere[co2], lca.dicts.activity[carbon_fibre]]
    )

scores = np.array(scores)
ng_values = np.array(ng_values)
cf_values = np.array(cf_values)
co2_values = np.array(co2_values)

# Pearson correlation coefficient between each parameter and the LCIA score
r_ng = np.corrcoef(ng_values, scores)[0, 1]
r_cf = np.corrcoef(cf_values, scores)[0, 1]
r_co2 = np.corrcoef(co2_values, scores)[0, 1]

print("Correlation with LCIA score:")
print(f"  natural_gas -> CF production (237 MJ/kg):   r = {r_ng:.3f}")
print(f"  CF -> bike production (2.5 kg CF):           r = {r_cf:.3f}")
print(f"  CO2 emission from CF production (26.6 kg):  r = {r_co2:.3f}")
print()
print("The dominant driver is the one with the highest absolute correlation.")

## Correlating exchanges (from main notebook)

In [ ]:
ng_samples = np.random.triangular(200, 237, 300, size=100)
co2_samples = 26.6 / 237 * ng_samples * np.random.normal(loc=1, scale=0.025, size=100)

dp_correlated = bwp.create_datapackage(seed=42)

dp_correlated.add_persistent_array(
    matrix='technosphere_matrix',
    indices_array=np.array([(natural_gas, carbon_fibre)], dtype=bwp.INDICES_DTYPE),
    data_array=ng_samples.reshape((1, -1)),
    flip_array=np.array([True]),
)
dp_correlated.add_persistent_array(
    matrix='biosphere_matrix',
    indices_array=np.array([(co2, carbon_fibre)], dtype=bwp.INDICES_DTYPE),
    data_array=co2_samples.reshape((1, -1)),
)

## Exercise 8 — Add a third scenario option

In [ ]:
# Add 2.0 kg CF as a "medium" option alongside 2.5 (original) and 1.5 (lightweight)
# The cf scenario array now has 3 columns: [2.5, 2.0, 1.5]
# The double bike array still has 2 columns: [1, 2]
# combinatorial=True gives 3 × 2 = 6 scenarios

dp_scenarios_3 = bwp.create_datapackage(combinatorial=True)

dp_scenarios_3.add_persistent_array(
    matrix='technosphere_matrix',
    indices_array=np.array([(carbon_fibre, bike)], dtype=bwp.INDICES_DTYPE),
    data_array=np.array([(2.5, 2.0, 1.5)]),
    flip_array=np.array([True]),
    name='cf scenario'
)
dp_scenarios_3.add_persistent_array(
    matrix='technosphere_matrix',
    indices_array=np.array([(bike, bike)], dtype=bwp.INDICES_DTYPE),
    data_array=np.array([(1, 2)]),
    name='double bike'
)

# 3 CF options × 2 bike options = 6 combinations
scenario_mapping_3 = {
    (0, 0): "Original CF (2.5 kg), single bike",
    (0, 1): "Original CF (2.5 kg), each counts double",
    (1, 0): "Medium CF (2.0 kg), single bike",
    (1, 1): "Medium CF (2.0 kg), each counts double",
    (2, 0): "Lightweight CF (1.5 kg), single bike",
    (2, 1): "Lightweight CF (1.5 kg), each counts double",
}

lca = bc.LCA(
    demand={bike: 1},
    data_objs=[dp_static, dp_scenarios_3],
    use_arrays=True,
)
lca.lci()
lca.lcia()

resource_group = next(grp for grp in lca.technosphere_mm.groups if grp.label == 'double bike').indexer.indexer

print(f"{lca.score:.2f} kg CO2-eq | {scenario_mapping_3[resource_group.index]}")

for _ in lca:
    print(f"{lca.score:.2f} kg CO2-eq | {scenario_mapping_3[resource_group.index]}")

# The "medium, counts double" scenario is index (1, 1)
# Score should be roughly: 26.6 kg CO2/kg CF × 2.0 kg CF / 2 bikes = 26.6 kg CO2-eq

## Exercise 9 — Benchmark: new LCA vs. redo_lci

In [ ]:
import time

demands = [
    {bike: 1},
    {carbon_fibre: 1},
    {natural_gas: 1},
]

# Approach (a): new LCA object each time — rebuilds A and factorises it each call
t0 = time.perf_counter()
scores_a = []
for demand in demands:
    lca_new = bc.LCA(demand=demand, data_objs=[dp_static])
    lca_new.lci()
    lca_new.lcia()
    scores_a.append(lca_new.score)
t1 = time.perf_counter()
print(f"New LCA each time: {(t1 - t0)*1000:.1f} ms  scores={[f'{s:.1f}' for s in scores_a]}")

# Approach (b): redo_lci — reuses LU factorisation, only solves new right-hand side
lca_reuse = bc.LCA(demand=demands[0], data_objs=[dp_static])
lca_reuse.lci()
lca_reuse.lcia()

t2 = time.perf_counter()
scores_b = [lca_reuse.score]
for demand in demands[1:]:
    lca_reuse.redo_lci(demand)
    lca_reuse.lcia()
    scores_b.append(lca_reuse.score)
t3 = time.perf_counter()
print(f"Reuse LCA object:  {(t3 - t2)*1000:.1f} ms  scores={[f'{s:.1f}' for s in scores_b]}")

print(f"\nSpeedup: {(t1-t0)/(t3-t2):.1f}×")
print("(With a real database of 20,000 activities, the speedup is much larger.")